In [0]:
import sys
sys.path.append('/Workspace/Users/saik84328@gmail.com')

from pyspark.sql import functions as F
from delta.tables import DeltaTable
from SetUp.Config import bronze_schema, silver_schema, gold_schema
from pyspark.sql.types import *
from datetime import datetime
import uuid


In [0]:


start_time = datetime.now()

print(start_time)

In [0]:
%run /Workspace/Users/saik84328@gmail.com/DataBricksLearning/AuditData

In [0]:
Gold_df = spark.table("helathcare_silver.helathcare_silver.SL_PatienatInfo")
display(Gold_df)

In [0]:
#Show hospital-wise revenue and patient count.
hospital_revenue_df = (
    Gold_df
    .groupBy("Hosiptalname")
    .agg(
        F.sum("Revenue").alias("Total_Revenue"),
        F.countDistinct("Patient_id").alias("Total_Patients"),
        F.avg("Revenue").alias("Avg_Revenue_Amount")
    )
)
display(hospital_revenue_df)

In [0]:
hospital_revenue_df.write.format("delta") \
.mode("overwrite") \
.saveAsTable(
    "helathcare_gold.gold_tables.gold_hospital_revenue"
)

In [0]:
#Which insurance company pays the highest claims?
insurance_summary_df = (
    Gold_df
    .groupBy("Insurance_provider")
    .agg(
        F.sum("Cliams_amount").alias("Total_Claims"),
        F.countDistinct("Patient_id").alias("Patient_Count")
    )
)
display(insurance_summary_df)

In [0]:
insurance_summary_df.write.format("delta") \
.mode("overwrite") \
.saveAsTable(
    "helathcare_gold.gold_tables.gold_hospital_revenuey"
)

In [0]:
#Show patient distribution by gender and marital status.
patient_demographics_df = (
    Gold_df
    .groupBy(
        "Gender",
        "Marital"
    )
    .agg(
        F.countDistinct("patient_id")
        .alias("Patient_Count")
    )
)
display(patient_demographics_df)

In [0]:
#Show monthly revenue trend.
claims_trend_df = (
    Gold_df
    .withColumn(
        "Claim_Month",
        F.date_format(
            "Todate",
            "yyyy-MM"
        )
    )
    .groupBy("Claim_Month")
    .agg(
        F.sum("Revenue")
        .alias("Monthly_Revenue")
    )
)
display(claims_trend_df)

In [0]:
claims_trend_df.write.format("delta") \
.mode("overwrite") \
.saveAsTable(
    "helathcare_gold.gold_tables.gold_monthly_claims"
)

In [0]:
#Which hospitals serve the most patients?
top_hospital_df = (
    Gold_df
    .groupBy("Hosiptalname")
    .agg(
        F.countDistinct("Patient_id")
        .alias("Patient_Count")
    )
    .orderBy(
        F.desc("Patient_Count")
    )
)
display(top_hospital_df)

In [0]:
top_hospital_df.write.format("delta") \
.mode("overwrite") \
.saveAsTable(
    "helathcare_gold.gold_tables.gold_patient_demographics"
)

In [0]:
end_time = datetime.now()

duration_seconds = int(
    (end_time - start_time).total_seconds()
)

print(duration_seconds)

In [0]:
from pyspark.sql.types import LongType

# Target table

# Get Workflow Run ID
try:
    run_id = dbutils.jobs.taskContext().taskRunId()
except:
    run_id = f"MANUAL_{datetime.now().strftime('%Y%m%d%H%M%S')}"

target_table = "helathcare_gold.gold_tables.GL_PatienatInfo"
status = "SUCCESS"
error_message = None
notebook_name, table_name, layer = get_audit_metadata(target_table)
record_count = 0

try:
    Gold_df = spark.table("helathcare_silver.helathcare_silver.SL_PatienatInfo")
    
    record_count = spark.table("helathcare_silver.helathcare_silver.SL_PatienatInfo").count()
except Exception as e:
    status = "FAILED"
    error_message = str(e)
    raise
finally:
    end_time = datetime.now()
    
    from pyspark.sql.types import StructType, StructField, StringType, TimestampType
    audit_schema = StructType([
        StructField("run_id", StringType(), True),
        StructField("notebook_name", StringType(), True),
        StructField("layer", StringType(), True),
        StructField("table_name", StringType(), True),
        StructField("record_count", LongType(), True),
        StructField("start_time", TimestampType(), True),
        StructField("end_time", TimestampType(), True),
        StructField("duration_seconds", LongType(), True),
        StructField("status", StringType(), True),
        StructField("error_message", StringType(), True),
    ])
    audit_data = [(
        run_id,
        notebook_name,
        layer,
        table_name,
        record_count,
        start_time,
        end_time,
        duration_seconds,
        status,
        error_message,
    )]
    audit_df = spark.createDataFrame(audit_data, schema=audit_schema)
    audit_df = audit_df.withColumn("load_date", F.current_date())
    audit_df.write.format("delta").mode("append").saveAsTable("helathcare_audit.audit_table.Patient_load_details")